In [2]:
# ============================================================
# RASTERIZE NUMERIC POLYGON ATTRIBUTE TO ALIGNED COG
#
# Input polygon:
#   BergFlate250_DannelsesAlder2.shp
#
# Numeric field:
#   LogALder
#
# Output:
#   geochronology_LogALder.tif
# ============================================================

import arcpy
import os
import time
from arcpy.sa import ExtractByMask

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ------------------------------------------------------------
# USER INPUTS
# ------------------------------------------------------------

mask_raster = r"E:\mask\Mask_land_Kalk_cog.tif"

input_shp = (
    r"N:\Prosjekter\411200_Kalk_i_grunnen\Berggrunn_N250_derivates"
    r"\BergFlate250_DannelsesAlder2.shp"
)

value_field = "LogAlder"

out_dir = r"G:\Geology_rasters_2026\Geochronology"
os.makedirs(out_dir, exist_ok=True)

out_cog = os.path.join(out_dir, "geochronology_LogALder.tif")

local_tmp_dir = r"C:\temp\rasterize_geochronology"
os.makedirs(local_tmp_dir, exist_ok=True)

tmp_gdb = os.path.join(local_tmp_dir, "temp_geochronology.gdb")

if arcpy.Exists(tmp_gdb):
    arcpy.management.Delete(tmp_gdb)

arcpy.management.CreateFileGDB(local_tmp_dir, "temp_geochronology.gdb")

# ------------------------------------------------------------
# ENVIRONMENT FROM MASK
# ------------------------------------------------------------

mask_desc = arcpy.Describe(mask_raster)

arcpy.env.workspace = tmp_gdb
arcpy.env.scratchWorkspace = tmp_gdb
arcpy.env.snapRaster = mask_raster
arcpy.env.cellSize = mask_raster
arcpy.env.extent = mask_raster
arcpy.env.outputCoordinateSystem = mask_desc.spatialReference
arcpy.env.mask = mask_raster

cell_size = float(
    arcpy.management.GetRasterProperties(
        mask_raster,
        "CELLSIZEX"
    ).getOutput(0)
)

print("Mask raster:", mask_raster)
print("CRS:", mask_desc.spatialReference.name)
print("Cell size:", cell_size)
print("Rows:", arcpy.management.GetRasterProperties(mask_raster, "ROWCOUNT").getOutput(0))
print("Cols:", arcpy.management.GetRasterProperties(mask_raster, "COLUMNCOUNT").getOutput(0))

# ------------------------------------------------------------
# CHECK INPUT
# ------------------------------------------------------------

if not arcpy.Exists(input_shp):
    raise FileNotFoundError(f"Input shapefile not found:\n{input_shp}")

fields = arcpy.ListFields(input_shp)
field_names = [f.name for f in fields]

if value_field not in field_names:
    print("\nAvailable fields:")
    for f in field_names:
        print(" -", f)

    raise ValueError(f"Field '{value_field}' not found.")

field_obj = [f for f in fields if f.name == value_field][0]

print("\nRasterizing numeric field:", value_field)
print("Field type:", field_obj.type)

if field_obj.type not in ["Double", "Single", "Integer", "SmallInteger"]:
    raise TypeError(
        f"Field '{value_field}' is not numeric. "
        f"Current type: {field_obj.type}"
    )

# ------------------------------------------------------------
# PROJECT TO MASK CRS IF NEEDED
# ------------------------------------------------------------

in_desc = arcpy.Describe(input_shp)
projected_fc = os.path.join(tmp_gdb, "geochronology_projected")

if in_desc.spatialReference.name != mask_desc.spatialReference.name:
    print("\nProjecting polygon layer to mask CRS...")

    arcpy.management.Project(
        in_dataset=input_shp,
        out_dataset=projected_fc,
        out_coor_system=mask_desc.spatialReference
    )

    work_fc = projected_fc

else:
    print("\nInput already has same CRS as mask.")
    work_fc = input_shp

# ------------------------------------------------------------
# COPY LOCALLY
# ------------------------------------------------------------

local_fc = os.path.join(tmp_gdb, "geochronology_local")

print("\nCopying polygons to local temporary geodatabase...")
arcpy.management.CopyFeatures(work_fc, local_fc)

# ------------------------------------------------------------
# RASTERIZE NUMERIC FIELD
# ------------------------------------------------------------

tmp_raster = os.path.join(tmp_gdb, "logalder_raster")

if arcpy.Exists(tmp_raster):
    arcpy.management.Delete(tmp_raster)

print("\nRasterizing LogALder polygons...")
start = time.time()

arcpy.conversion.PolygonToRaster(
    in_features=local_fc,
    value_field=value_field,
    out_rasterdataset=tmp_raster,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=cell_size
)

print(f"Rasterization completed in {(time.time() - start) / 60:.1f} min")

# ------------------------------------------------------------
# MASK ONCE
# ------------------------------------------------------------

print("\nMasking raster with land mask...")
start = time.time()

masked_raster = ExtractByMask(tmp_raster, mask_raster)

print(f"Masking completed in {(time.time() - start) / 60:.1f} min")

# ------------------------------------------------------------
# SAVE AS COG
# ------------------------------------------------------------

if arcpy.Exists(out_cog):
    arcpy.management.Delete(out_cog)

print("\nSaving aligned masked COG:")
print(out_cog)

start = time.time()

arcpy.management.CopyRaster(
    in_raster=masked_raster,
    out_rasterdataset=out_cog,
    pixel_type="32_BIT_FLOAT",
    nodata_value="-9999",
    format="COG"
)

print(f"COG saved in {(time.time() - start) / 60:.1f} min")

print("\nDone.")
print("Output raster:")
print(out_cog)

Mask raster: E:\mask\Mask_land_Kalk_cog.tif
CRS: ETRS_1989_UTM_Zone_33N
Cell size: 10.0
Rows: 162232
Cols: 135236

Rasterizing numeric field: LogAlder
Field type: Double

Projecting polygon layer to mask CRS...

Copying polygons to local temporary geodatabase...

Rasterizing LogALder polygons...
Rasterization completed in 6.0 min

Masking raster with land mask...
Masking completed in 16.1 min

Saving aligned masked COG:
G:\Geology_rasters_2026\Geochronology\geochronology_LogALder.tif
COG saved in 11.9 min

Done.
Output raster:
G:\Geology_rasters_2026\Geochronology\geochronology_LogALder.tif
